In [4]:
# Author: Giorgio Doglioni
# Purpose: compare one DWL wind profile with the nearest ICON model grid-cell profile.
# Notes added below explain each workflow step.
# Note: produced with support from GPT 5.5, which also cleaned it and provided comments. 
# Version 0, straight from the Hackaton in Trento (17-19 June 2026). Things to fix: awkward valid time selection, plot aesthetics, figure naming. 
# Disclaimer: a complete revision of the code is needed before using it for scientific purposes. 
"""
Clean DWL vs ICON single-profile comparison.
Hackathon
Assumptions:
- ICON grid longitude/latitude variables are: clon, clat
- ICON wind variables are GRIB shortName: u, v
- ICON vertical coordinate is derived from HHL and converted from MSL to AGL
- ICON terrain/orography is HSURF
- DWL wind speed variable is: wspeed(time, height)
- DWL wind direction variable is: wdir(time, height)
- DWL vertical coordinate is: height, above the sensor
- DWL sensor altitude variable is configured below; this script currently uses zsl
- DWL wdir is already meteorological wind-from-direction

No time averaging is performed.
The DWL profile is selected at the nearest time to the ICON valid time.

Only remaining user-defined quantities:
- If lon/lat are not variables in the LiDAR file, set lidar_lon and lidar_lat manually below.
"""

from pathlib import Path  # Object-oriented file paths.
import warnings       # Used to flag unusual but non-fatal vertical-coordinate cases.

import numpy as np
import xarray as xr              # Reads NetCDF files and GRIB files through cfgrib.
import matplotlib.pyplot as plt  # Creates terrain and profile figures.


# =============================================================================
# USER CONFIGURATION: adapt the lines below to your setup
# =============================================================================

date = "20250629"          # Case date in YYYYMMDD format; used for folder and filename construction.
iop = "sIOP7"              # Intensive observation period / case folder suffix.
lid_number = 999           # LiDAR identifier used in input/output filenames. Use 999 or another number for missing identifier. 
icon_hours = range(1, 12)  # ICON hours to process. Note: this currently processes 01..11 UTC.

downloads_root = Path("/Users/giorgio_doglioni/Downloads")  # Root folder for ICON case files and grid file.
kitcube_root = Path("/Users/giorgio_doglioni/dacepy/KITcube")  # Root folder for KITcube / LiDAR files.

BASE = downloads_root / f"{date}_{iop}"  # Case-specific ICON directory.

wl_file = kitcube_root / "WTX_20250629_600s_100m_wind_profile.nc"  # LiDAR NetCDF file to compare against ICON.
# Alternative filename pattern, if needed:
# wl_file = kitcube_root / f"WLS200s_{lid_number}_{date}_600s_100m_wind_profile.nc"
hhl_grib_file = BASE / "ilf3f00000000.grb2"  # Static/initial GRIB containing HHL and HSURF.
grid_file = downloads_root / "icon_grid_0060_R19B09_LN03.nc"  # ICON grid file with clon/clat cell centers.
# =============================================================================
# USER CONFIGURATION: adapt the lines above
# Lines below contain some plot settings that can be modified too. 
# =============================================================================

# ICON variables
icon_lon_name = "clon"
icon_lat_name = "clat"
icon_u_name = "u"
icon_v_name = "v"
icon_hhl_name = "HHL"
icon_hsurf_name = "HSURF"
icon_type_of_level = "generalVerticalLayer"

# DWL variables
wl_speed_name = "wspeed"
wl_direction_name = "wdir"
wl_height_name = "height"
wl_sensor_altitude_name = "zsl"  # Sensor altitude variable in the LiDAR file; check spelling against your NetCDF.

# LiDAR horizontal location.
# If your file contains lon/lat variables with these exact names, leave as below.
# If not, set lidar_lon and lidar_lat manually, e.g. lidar_lon = 11.12345
lidar_lon_name = "lon"
lidar_lat_name = "lat"
lidar_lon = None
lidar_lat = None

# Optional: manually force an ICON cell index. Keep None to use nearest clon/clat to LiDAR.
idx = None

# Plot settings
domain_km = 20.0  # Width/height of terrain map domain around the LiDAR, in km.
height_limits = (0, 3000)  # y-axis limits for profile plots, in metres.
save_figures = True  # Save PNG outputs to the current working directory.
show_figures = False  # Set True for interactive display instead of only saving figures.


# =============================================================================
# FILE OPENING
# =============================================================================

# Open the DWL NetCDF file. Kept as a small wrapper so file access is consistent.
def open_wl_dataset(path):
    return xr.open_dataset(path)


# Open the ICON grid NetCDF file containing cell-center longitude/latitude.
def open_icon_grid(path):
    return xr.open_dataset(path)


# Generic ICON GRIB reader: selects exactly one GRIB variable using cfgrib filter keys.
def open_icon_variable(grib_file, short_name, type_of_level=None, indexpath=""):
    """Open one ICON GRIB shortName with cfgrib."""
    # cfgrib can expose multiple GRIB messages in one file; filter_by_keys selects the desired one.
    filter_by_keys = {"shortName": short_name}
    if type_of_level is not None:
        filter_by_keys["typeOfLevel"] = type_of_level

    return xr.open_dataset(
        grib_file,
        engine="cfgrib",
        backend_kwargs={
            "filter_by_keys": filter_by_keys,
            "errors": "raise",
            "indexpath": indexpath,
        },
    )


# Convenience wrappers below make the later workflow easier to read.
def open_icon_u(grib_file):
    return open_icon_variable(
        grib_file,
        short_name=icon_u_name,
        type_of_level=icon_type_of_level,
    )


def open_icon_v(grib_file):
    return open_icon_variable(
        grib_file,
        short_name=icon_v_name,
        type_of_level=icon_type_of_level,
    )


def open_icon_hhl(grib_file):
    return open_icon_variable(
        grib_file,
        short_name=icon_hhl_name,
        type_of_level=None,
    )


def open_icon_hsurf(grib_file):
    return open_icon_variable(
        grib_file,
        short_name=icon_hsurf_name,
        type_of_level=None,
    )


# =============================================================================
# BASIC MATH / COORDINATES
# =============================================================================

# Convert horizontal wind components to scalar wind speed.
def wind_speed_from_uv(u, v):
    # Speed magnitude from orthogonal horizontal components.
    return np.sqrt(u**2 + v**2)


def wind_from_direction_from_uv(u, v):
    """
    Meteorological wind direction FROM which wind blows, clockwise from north.

    Assumes standard u/v flow components:
    u > 0 eastward, v > 0 northward.
    """
    # Convert mathematical vector angle to meteorological wind-from direction.
    return (270.0 - np.degrees(np.arctan2(v, u))) % 360.0


# Approximate local lon/lat differences as east/north distances in km.
# This is accurate enough for the small plotting/search domain used here.
def lonlat_to_km_offsets(lon, lat, lon0, lat0):
    earth_radius_km = 6371.0  # Mean Earth radius used for geographic distance approximations.
    lon = np.asarray(lon)
    lat = np.asarray(lat)

    x = np.deg2rad(lon - lon0) * earth_radius_km * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(lat - lat0) * earth_radius_km
    return x, y


# Great-circle distance between two lon/lat points; used for reporting the true grid-cell distance.
def haversine_km(lon1, lat1, lon2, lat2):
    r = 6371.0

    lon1 = np.deg2rad(lon1)
    lat1 = np.deg2rad(lat1)
    lon2 = np.deg2rad(lon2)
    lat2 = np.deg2rad(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return r * c


# Find the ICON cell whose center is closest to the LiDAR location.
def nearest_icon_index(lon_icon, lat_icon, target_lon, target_lat):
    x, y = lonlat_to_km_offsets(lon_icon, lat_icon, target_lon, target_lat)
    return int(np.nanargmin(x**2 + y**2))


# =============================================================================
# EXPLICIT DATA READERS
# =============================================================================

# Read and validate ICON cell-center coordinates.
def read_icon_grid_lon_lat(grid_file):
    """Read ICON clon/clat, converting radians to degrees if needed."""
    grid = open_icon_grid(grid_file)

    if icon_lon_name not in grid:
        raise KeyError(f"ICON grid variable {icon_lon_name!r} not found. Available: {list(grid.variables)}")
    if icon_lat_name not in grid:
        raise KeyError(f"ICON grid variable {icon_lat_name!r} not found. Available: {list(grid.variables)}")

    # Convert xarray DataArrays to plain NumPy arrays and remove length-1 dimensions.
    lon = np.asarray(grid[icon_lon_name].values).squeeze()
    lat = np.asarray(grid[icon_lat_name].values).squeeze()

    if lon.ndim != 1 or lat.ndim != 1:
        raise ValueError(f"Expected 1D ICON clon/clat. Got lon={lon.shape}, lat={lat.shape}")

    # ICON clon/clat are commonly radians.
    # If values look like radians, convert to degrees for distance calculations and plotting.
    if np.nanmax(np.abs(lon)) <= 2 * np.pi and np.nanmax(np.abs(lat)) <= np.pi:
        lon = np.rad2deg(lon)
        lat = np.rad2deg(lat)

    return lon, lat, grid


# Read the LiDAR position either from manual configuration or exact lon/lat variables.
def read_lidar_location(ds_wl):
    """
    Read LiDAR lon/lat.

    No guessing:
    - If lidar_lon and lidar_lat are set manually, use those.
    - Otherwise require variables/coords named exactly lidar_lon_name/lidar_lat_name.
    """
    if lidar_lon is not None and lidar_lat is not None:
        return float(lidar_lon), float(lidar_lat), "manual config"

    if lidar_lon_name not in ds_wl.variables:
        raise KeyError(
            f"LiDAR longitude not found as {lidar_lon_name!r}. "
            f"Set lidar_lon manually, or change lidar_lon_name. "
            f"Available variables: {list(ds_wl.variables)}"
        )

    if lidar_lat_name not in ds_wl.variables:
        raise KeyError(
            f"LiDAR latitude not found as {lidar_lat_name!r}. "
            f"Set lidar_lat manually, or change lidar_lat_name. "
            f"Available variables: {list(ds_wl.variables)}"
        )

    lon = float(np.asarray(ds_wl[lidar_lon_name].values).squeeze())
    lat = float(np.asarray(ds_wl[lidar_lat_name].values).squeeze())

    return lon, lat, f"variables {lidar_lon_name!r}/{lidar_lat_name!r}"


# Read the LiDAR sensor altitude; needed to convert LiDAR heights above sensor to MSL.
def read_lidar_sensor_altitude(ds_wl):
    """Read zls: altitude of the LiDAR sensor above mean sea level."""
    if wl_sensor_altitude_name not in ds_wl.variables:
        raise KeyError(
            f"LiDAR sensor altitude variable {wl_sensor_altitude_name!r} not found. "
            f"Available variables: {list(ds_wl.variables)}"
        )

    zls = float(np.asarray(ds_wl[wl_sensor_altitude_name].values).squeeze())
    return zls


# Read ICON terrain height/orography at every grid cell.
def read_hsurf(hhl_grib_file):
    ds_hsurf = open_icon_hsurf(hhl_grib_file)

    if icon_hsurf_name not in ds_hsurf:
        raise KeyError(f"{icon_hsurf_name!r} not found. Available: {list(ds_hsurf.data_vars)}")

    hsurf = ds_hsurf[icon_hsurf_name].squeeze()

    if "values" not in hsurf.dims:
        raise ValueError(f"Expected HSURF dimension 'values'. Got {hsurf.dims}")

    return np.asarray(hsurf.values, dtype=float), ds_hsurf


# =============================================================================
# PROFILE EXTRACTION
# =============================================================================

# Select one LiDAR wind-speed and wind-direction profile nearest to the requested time.
def extract_lidar_profile(wl_file, valid_time):
    """
    Extract DWL/LiDAR wspeed/wdir profile at nearest time.

    LiDAR height is above the sensor. To compare with ICON AGL over model terrain,
    the plotted LiDAR vertical coordinate remains height above sensor.
    The script also computes height_msl = zls + height for diagnostics.
    """
    ds = open_wl_dataset(wl_file)

    required = [wl_speed_name, wl_direction_name, wl_height_name, wl_sensor_altitude_name]
    missing = [name for name in required if name not in ds.variables]
    if missing:
        raise KeyError(f"Missing required LiDAR variables: {missing}. Available: {list(ds.variables)}")

    speed_da = ds[wl_speed_name]
    direction_da = ds[wl_direction_name]
    height_da = ds[wl_height_name]

    if speed_da.dims != direction_da.dims:
        raise ValueError(
            f"{wl_speed_name} and {wl_direction_name} must have identical dims. "
            f"Got {speed_da.dims} and {direction_da.dims}."
        )

    if speed_da.dims != ("time", "height"):
        raise ValueError(
            f"Expected {wl_speed_name} dims exactly ('time', 'height'). "
            f"Got {speed_da.dims}."
        )

    # Select the LiDAR profile closest in time to the requested comparison time.
    selected_speed = speed_da.sel(time=np.datetime64(valid_time), method="nearest")
    selected_direction = direction_da.sel(time=np.datetime64(valid_time), method="nearest")
    selected_time = selected_speed["time"].values

    height = np.asarray(height_da.values, dtype=float).squeeze()
    speed = np.asarray(selected_speed.values, dtype=float).squeeze()
    # Normalize wind directions to the 0..360 degree interval.
    direction = np.asarray(selected_direction.values, dtype=float).squeeze() % 360.0

    if height.ndim != 1 or speed.ndim != 1 or direction.ndim != 1:
        raise ValueError(
            f"Expected 1D LiDAR profiles. Got height={height.shape}, "
            f"speed={speed.shape}, direction={direction.shape}."
        )

    if not (len(height) == len(speed) == len(direction)):
        raise ValueError(
            f"LiDAR length mismatch: height={len(height)}, speed={len(speed)}, direction={len(direction)}"
        )

    zls = read_lidar_sensor_altitude(ds)
    height_msl = height + zls

    # Sort by height so plotted profiles always increase upward.
    order = np.argsort(height)

    return {
        "height": height[order],              # above sensor
        "height_msl": height_msl[order],      # above mean sea level
        "speed": speed[order],
        "direction": direction[order],
        "selected_time": str(selected_time),
        "zls": zls,
        "ds": ds,
    }


# Extract ICON u/v winds and HHL-derived heights at a single grid-cell index.
def extract_icon_profile(icon_grib_file, hhl_grib_file, idx):
    """Extract ICON u/v profile and HHL-derived heights at one ICON cell."""
    ds_u = open_icon_u(icon_grib_file)
    ds_v = open_icon_v(icon_grib_file)
    ds_hhl = open_icon_hhl(hhl_grib_file)
    hsurf, ds_hsurf = read_hsurf(hhl_grib_file)

    for varname, ds in [(icon_u_name, ds_u), (icon_v_name, ds_v), (icon_hhl_name, ds_hhl)]:
        if varname not in ds:
            raise KeyError(f"{varname!r} not found. Available: {list(ds.data_vars)}")

    u_da = ds_u[icon_u_name].squeeze()
    v_da = ds_v[icon_v_name].squeeze()
    hhl_da = ds_hhl[icon_hhl_name].squeeze()

    for name, da in [(icon_u_name, u_da), (icon_v_name, v_da), (icon_hhl_name, hhl_da)]:
        if "values" not in da.dims:
            raise ValueError(f"Expected {name!r} to have dimension 'values'. Got {da.dims}")

    # Extract the vertical column at the selected ICON horizontal cell.
    u = np.asarray(u_da.isel(values=idx).values, dtype=float).squeeze()
    v = np.asarray(v_da.isel(values=idx).values, dtype=float).squeeze()
    hhl_half_msl = np.asarray(hhl_da.isel(values=idx).values, dtype=float).squeeze()

    terrain = float(hsurf[idx])

    # HHL is usually on half levels; average adjacent half levels to get full model-level heights.
    if hhl_half_msl.size == u.size + 1:
        height_msl = 0.5 * (hhl_half_msl[:-1] + hhl_half_msl[1:])
    elif hhl_half_msl.size == u.size:
        warnings.warn("HHL has same size as u/v; using HHL directly as model-level height.")
        height_msl = hhl_half_msl
    else:
        raise ValueError(
            f"Vertical mismatch: len(u)={u.size}, len(v)={v.size}, len(HHL)={hhl_half_msl.size}"
        )

    # Convert model-level height from mean sea level to above-ground level over ICON terrain.
    height_agl = height_msl - terrain

    speed = wind_speed_from_uv(u, v)
    direction = wind_from_direction_from_uv(u, v)

    order = np.argsort(height_agl)

    return {
        "height": height_agl[order],       # above ICON terrain
        "height_agl": height_agl[order],
        "height_msl": height_msl[order],
        "terrain": terrain,
        "u": u[order],
        "v": v[order],
        "speed": speed[order],
        "direction": direction[order],
        "ds_u": ds_u,
        "ds_v": ds_v,
        "ds_hhl": ds_hhl,
        "ds_hsurf": ds_hsurf,
    }


# Return the ICON valid_time coordinate if cfgrib exposed it.
def icon_valid_time(ds_u):
    if "valid_time" in ds_u:
        return str(np.asarray(ds_u["valid_time"].values).squeeze())
    return "valid_time not found"


def read_icon_valid_time(ds_u):
    """Read the actual ICON valid_time as a NumPy datetime64 value.

    The hour encoded in the filename is treated only as a file/forecast-hour label.
    The timestamp used for LiDAR selection and figure titles comes from the GRIB
    metadata itself. This avoids manual offsets such as +12 h.
    """
    if "valid_time" not in ds_u:
        raise KeyError(
            "ICON valid_time not found in the GRIB metadata. "
            "Cannot safely align LiDAR and ICON times without guessing."
        )

    return np.asarray(ds_u["valid_time"].values).squeeze()


def format_datetime_for_title(value):
    """Format a NumPy datetime64 for human-readable plot titles/log messages."""
    return np.datetime_as_string(np.datetime64(value), unit="m").replace("T", " ") + " UTC"


def format_datetime_for_filename(value):
    """Format a NumPy datetime64 into a filesystem-friendly UTC timestamp."""
    stamp = np.datetime_as_string(np.datetime64(value), unit="m")
    return stamp.replace("-", "").replace(":", "").replace("T", "_") + "UTC"


# =============================================================================
# PLOTTING
# =============================================================================

# Plot local ICON terrain around the LiDAR and mark the selected nearest ICON grid cell.
def plot_terrain_map(lon_icon, lat_icon, hsurf, lidar_lon, lidar_lat, idx, output_path=None):
    # Express ICON grid-cell locations relative to the LiDAR for local plotting.
    x, y = lonlat_to_km_offsets(lon_icon, lat_icon, lidar_lon, lidar_lat)

    half = domain_km / 2.0
    # Keep only grid cells inside the requested local map window and with finite terrain data.
    mask = (
        (x >= -half)
        & (x <= half)
        & (y >= -half)
        & (y <= half)
        & np.isfinite(hsurf)
    )

    x_nearest, y_nearest = lonlat_to_km_offsets(lon_icon[idx], lat_icon[idx], lidar_lon, lidar_lat)
    distance = haversine_km(lidar_lon, lidar_lat, lon_icon[idx], lat_icon[idx])

    fig, ax = plt.subplots(figsize=(8, 7))
    terrain = ax.tricontourf(x[mask], y[mask], hsurf[mask], levels=20, cmap="terrain")
    cbar = fig.colorbar(terrain, ax=ax, shrink=0.85)
    cbar.set_label("ICON HSURF [m]")

    ax.scatter(0, 0, marker="*", s=180, c="red", edgecolor="black", label="LiDAR", zorder=5)
    ax.scatter(x_nearest, y_nearest, marker="x", s=130, c="black", linewidth=2, label=f"ICON values={idx}", zorder=6)

    ax.set_xlim(-half, half)
    ax.set_ylim(-half, half)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("East-west distance from LiDAR [km]")
    ax.set_ylabel("North-south distance from LiDAR [km]")
    ax.set_title(
        f"ICON terrain around LiDAR\n"
        f"distance={distance:.3f} km, ICON HSURF={hsurf[idx]:.1f} m"
    )
    ax.legend()

    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")

    return fig, ax


# Plot LiDAR and ICON wind profiles side-by-side: speed and meteorological direction.
def plot_profiles(lidar_profile, icon_profile, comparison_time_label, idx, output_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

    axes[0].plot(lidar_profile["speed"], lidar_profile["height"], marker="o", label="DWL")
    axes[0].plot(icon_profile["speed"], icon_profile["height_agl"], marker="s", label=f"ICON values={idx}")
    axes[0].set_xlabel("Wind speed [m s$^{-1}$]")
    axes[0].set_ylabel("Height [m]\nDWL: above sensor; ICON: above ICON terrain")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(lidar_profile["direction"], lidar_profile["height"], marker="o", label="DWL")
    axes[1].plot(icon_profile["direction"], icon_profile["height_agl"], marker="s", label=f"ICON values={idx}")
    axes[1].set_xlabel("Wind direction FROM [deg]")
    axes[1].set_xlim(0, 360)
    axes[1].set_xticks(np.arange(0, 361, 60))
    axes[1].grid(True)
    axes[1].legend()

    if height_limits is not None:
        for ax in axes:
            ax.set_ylim(*height_limits)

    fig.suptitle(f"DWL/LiDAR vs ICON profile at {comparison_time_label}")
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")

    return fig, axes


# =============================================================================
# ONE-HOUR WORKFLOW
# =============================================================================

# Complete workflow for one ICON forecast file/hour.
def run_one_hour(icon_hour, plot_terrain=False):
    # Build the hour-specific ICON GRIB filename.
    # Important: icon_hour is only the file/forecast-hour label. The real
    # comparison timestamp is read from the ICON valid_time metadata below.
    icon_grib_file = BASE / f"ilf3f00{icon_hour:02d}0000.grb2"

    # Open ICON u first so we can read the actual GRIB valid_time.
    ds_u = open_icon_u(icon_grib_file)
    comparison_time = read_icon_valid_time(ds_u)
    comparison_time_iso = np.datetime_as_string(np.datetime64(comparison_time), unit="s")
    comparison_time_label = format_datetime_for_title(comparison_time)

    # Put the real valid time in output filenames. Keep fXX so you can still
    # trace which ICON file produced the figure.
    output_prefix = f"wl_{lid_number}_icon_{format_datetime_for_filename(comparison_time)}_f{icon_hour:02d}"

    print("\n" + "=" * 80)
    print("ICON file hour/lead label:", icon_hour)
    print("ICON valid/comparison time:", comparison_time_label)
    print("WL file:", wl_file)
    print("ICON file:", icon_grib_file)
    print("=" * 80)

    ds_wl = open_wl_dataset(wl_file)
    lidar_lon_value, lidar_lat_value, lidar_location_source = read_lidar_location(ds_wl)
    zls = read_lidar_sensor_altitude(ds_wl)

    lon_icon, lat_icon, grid = read_icon_grid_lon_lat(grid_file)
    
    print("ICON u values:", ds_u["u"].sizes["values"])
    print("ICON grid cells:", lon_icon.size)
    
    # Safety check: the GRIB field must have the same number of cells as the grid file.
    if ds_u["u"].sizes["values"] != lon_icon.size:
        raise ValueError(
            f"ICON GRIB values dimension and grid file size do not match: "
            f"u values={ds_u['u'].sizes['values']}, grid cells={lon_icon.size}. "
            "You may be using the wrong ICON grid file."
        )
    hsurf, ds_hsurf = read_hsurf(hhl_grib_file)

    if hsurf.size != lon_icon.size:
        raise ValueError(f"ICON grid/HSURF size mismatch: clon={lon_icon.size}, HSURF={hsurf.size}")

    selected_idx = idx
    # If no manual ICON cell index was provided, choose the nearest grid cell automatically.
    if selected_idx is None:
        selected_idx = nearest_icon_index(lon_icon, lat_icon, lidar_lon_value, lidar_lat_value)

    distance = haversine_km(
        lidar_lon_value,
        lidar_lat_value,
        lon_icon[selected_idx],
        lat_icon[selected_idx],
    )

    lidar_profile = extract_lidar_profile(wl_file, comparison_time_iso)
    icon_profile = extract_icon_profile(icon_grib_file, hhl_grib_file, selected_idx)

    terrain_fig = terrain_ax = None
    # Plot terrain only when requested; the main loop requests it for the first hour only.
    if plot_terrain:
        terrain_path = f"{output_prefix}_terrain.png" if save_figures else None
        terrain_fig, terrain_ax = plot_terrain_map(
            lon_icon,
            lat_icon,
            hsurf,
            lidar_lon_value,
            lidar_lat_value,
            selected_idx,
            output_path=terrain_path,
        )

    profile_path = f"{output_prefix}_profile.png" if save_figures else None
    profile_fig, profile_axes = plot_profiles(
        lidar_profile,
        icon_profile,
        comparison_time_label,
        selected_idx,
        output_path=profile_path,
    )

    print("\n--- Location ---")
    print("LiDAR lon/lat:", lidar_lon_value, lidar_lat_value, "| source:", lidar_location_source)
    print("LiDAR zls [m MSL]:", zls)
    print("ICON idx:", selected_idx)
    print("ICON lon/lat:", lon_icon[selected_idx], lat_icon[selected_idx])
    print("Distance [km]:", distance)
    print("ICON HSURF [m MSL]:", hsurf[selected_idx])
    print("zls - ICON HSURF [m]:", zls - hsurf[selected_idx])

    print("\n--- Time ---")
    print("ICON file hour/lead label:", icon_hour)
    print("ICON valid/comparison time:", comparison_time_label)
    print("LiDAR target comparison time:", comparison_time_iso)
    print("LiDAR selected time:", lidar_profile["selected_time"])
    print("ICON valid time from profile dataset:", icon_valid_time(icon_profile["ds_u"]))

    print("\n--- Heights ---")
    print("LiDAR height above sensor [m]:", np.nanmin(lidar_profile["height"]), np.nanmax(lidar_profile["height"]))
    print("LiDAR height MSL [m]:", np.nanmin(lidar_profile["height_msl"]), np.nanmax(lidar_profile["height_msl"]))
    print("ICON height AGL [m]:", np.nanmin(icon_profile["height_agl"]), np.nanmax(icon_profile["height_agl"]))
    print("ICON height MSL [m]:", np.nanmin(icon_profile["height_msl"]), np.nanmax(icon_profile["height_msl"]))

    print("\n--- Variables ---")
    print("LiDAR speed:", wl_speed_name)
    print("LiDAR direction:", wl_direction_name, "(wind_from_direction)")
    print("LiDAR height:", wl_height_name, "(above sensor)")
    print("LiDAR sensor altitude:", wl_sensor_altitude_name)
    print("ICON u/v:", icon_u_name, icon_v_name)
    print("ICON grid lon/lat:", icon_lon_name, icon_lat_name)
    print("ICON vertical/terrain:", icon_hhl_name, icon_hsurf_name)

    # Report generated figure names when files were saved.
    if save_figures:
        print("\nSaved:", profile_path)
        if plot_terrain:
            print("Saved:", terrain_path)

    return {
        "idx": selected_idx,
        "lidar_profile": lidar_profile,
        "icon_profile": icon_profile,
        "profile_fig": profile_fig,
        "profile_axes": profile_axes,
        "terrain_fig": terrain_fig,
        "terrain_ax": terrain_ax,
        "distance_km": distance,
        "lidar_lon": lidar_lon_value,
        "lidar_lat": lidar_lat_value,
        "icon_lon": lon_icon[selected_idx],
        "icon_lat": lat_icon[selected_idx],
        "icon_hsurf": hsurf[selected_idx],
        "zls": zls,
    }


# =============================================================================
# RUN
# =============================================================================

# Run the comparison only when this file is executed as a script, not when imported.
if __name__ == "__main__":
    first_hour = list(icon_hours)[0]  # Used to plot the terrain map only once.

    # Process each requested ICON hour.
    for icon_hour in icon_hours:
        # Store the returned diagnostics in case you run this interactively and want to inspect the last hour.
        result = run_one_hour(
            icon_hour,
            plot_terrain=(icon_hour == first_hour),
        )

        # Close figures after saving to avoid accumulating many open matplotlib figures.
        if not show_figures:
            plt.close("all")

    if show_figures:
        plt.show()



ICON file hour/lead label: 1
ICON valid/comparison time: 2025-06-29 13:00 UTC
WL file: /Users/giorgio_doglioni/dacepy/KITcube/WTX_20250629_600s_100m_wind_profile.nc
ICON file: /Users/giorgio_doglioni/Downloads/20250629_sIOP7/ilf3f00010000.grb2
ICON u values: 945940
ICON grid cells: 945940

--- Location ---
LiDAR lon/lat: 11.32158 46.45523 | source: variables 'lon'/'lat'
LiDAR zls [m MSL]: 235.0
ICON idx: 285118
ICON lon/lat: 11.323938990001439 46.45463800681965
Distance [km]: 0.1923261983620162
ICON HSURF [m MSL]: 229.5
zls - ICON HSURF [m]: 5.5

--- Time ---
ICON file hour/lead label: 1
ICON valid/comparison time: 2025-06-29 13:00 UTC
LiDAR target comparison time: 2025-06-29T13:00:00
LiDAR selected time: 2025-06-29T13:00:00.000000000
ICON valid time from profile dataset: 2025-06-29T13:00:00.000000000

--- Heights ---
LiDAR height above sensor [m]: 0.0 5000.0
LiDAR height MSL [m]: 235.0 5235.0
ICON height AGL [m]: 10.0 20471.42578125
ICON height MSL [m]: 239.5 20700.92578125

--- Vari